# Hyper Instancing Meshes in a BIM Model - 2

In part 1 we were able to successfully create a transformation matrix for our shape and replace our representations in the model with this new one. However, we were unable to make the scale factor adequately work for us. Here, we try to create a [mapped representation](https://docs.ifcopenshell.org/ifcopenshell-python/geometry_creation.html#types-and-mapped-representations) and assign it to a type in the model. Let's see what we can do.

In [1]:
import ifcopenshell
import pymeshlab
import pandas as pd
import numpy as np
import ifcopenshell.geom
import ifcopenshell.util.shape
import ifcopenshell.util.representation
import ifcopenshell.api.project

In [2]:
# Loading our unit pipe
ms = pymeshlab.MeshSet()
ms.load_new_mesh('data/simple_pipe.obj')

m = ms.current_mesh()
v_matrix = m.vertex_matrix()

f_matrix = m.face_matrix()

In [3]:
def get_transformation_matrix(input_v):
    '''
    Given a vertex array, finds the transformation matrix required to rotate, scale, and translate a shape into that position. Works best for pipes.

    inputs:
        input_v: numpy array

    returns:
        transformation_matrix: numpy array 
    '''
    # Step 1: Center vertices
    centroid = np.min(input_v, axis=0) + (np.max(input_v, axis=0) - np.min(input_v, axis=0))/2
    centered_v = input_v - centroid

    # Step 2: Find covariance matrix
    cov_mat = np.cov(centered_v, rowvar=False)

    # Step 3: Find eigenvectors and values of the cov matrix
    eg_val, eg_vec = np.linalg.eigh(cov_mat)
    eg_val = np.round(eg_val, 4)

    # Step 4: Reorder eigenvector matrix to match length axis
    length_axis = None
    minor_axis_1 = None
    minor_axis_2 = None

    if eg_val[0] == eg_val[1]: 
        length_axis = 2
        minor_axis_1 = 0
        minor_axis_2 = 1

    elif eg_val[0] == eg_val[2]:
        length_axis = 1
        minor_axis_1 = 0
        minor_axis_2 = 2

    elif eg_val[1] == eg_val[2]:
        length_axis = 0
        minor_axis_1 = 2
        minor_axis_2 = 1

    else:
        length_axis = np.nan
        minor_axis_1 = np.nan
        minor_axis_2 = np.nan

    eg_vec = eg_vec[:, [minor_axis_1, length_axis, minor_axis_2]]

    # Step 5: Find scale vector by projecting original verts onto the eg_vectors
    projected_v = centered_v @ eg_vec

    scale_vec = np.max(projected_v, axis = 0)

    # Step 6: Find the final transformation matrix
    T_matrix = np.eye(4, 4)
    T_matrix[:3, :3] = eg_vec @ np.diag(scale_vec)
    T_matrix[:3, 3] = centroid

    # Return

    return T_matrix

In [4]:
def get_transformation_matrix_2(input_v):
    '''
    Given a vertex array, finds the transformation matrix required to rotate, scale, and translate a shape into that position. Works best for pipes.

    inputs:
        input_v: numpy array

    returns:
        transformation_matrix: numpy array
        scale_vec: numpy array
    '''
    # Step 1: Center vertices
    centroid = np.min(input_v, axis=0) + (np.max(input_v, axis=0) - np.min(input_v, axis=0))/2
    centered_v = input_v - centroid

    # Step 2: Find covariance matrix
    cov_mat = np.cov(centered_v, rowvar=False)

    # Step 3: Find eigenvectors and values of the cov matrix
    eg_val, eg_vec = np.linalg.eigh(cov_mat)
    eg_val = np.round(eg_val, 4)

    # Step 4: Reorder eigenvector matrix to match length axis
    length_axis = None
    minor_axis_1 = None
    minor_axis_2 = None

    if eg_val[0] == eg_val[1]: 
        length_axis = 2
        minor_axis_1 = 0
        minor_axis_2 = 1

    elif eg_val[0] == eg_val[2]:
        length_axis = 1
        minor_axis_1 = 0
        minor_axis_2 = 2

    elif eg_val[1] == eg_val[2]:
        length_axis = 0
        minor_axis_1 = 2
        minor_axis_2 = 1

    else:
        length_axis = np.nan
        minor_axis_1 = np.nan
        minor_axis_2 = np.nan

    eg_vec = eg_vec[:, [minor_axis_1, length_axis, minor_axis_2]]

    # Step 5: Find scale vector by projecting original verts onto the eg_vectors
    projected_v = centered_v @ eg_vec

    scale_vec = np.max(projected_v, axis = 0)

    # Step 6: Find the final transformation matrix
    T_matrix = np.eye(4, 4)
    T_matrix[:3, :3] = eg_vec
    T_matrix[:3, 3] = centroid

    # Return

    return ( T_matrix, scale_vec )

In [5]:
model = ifcopenshell.open('data/O-S1-INS-Plumbing - Sanitair.ifc')

In [6]:
pipes = model.by_type("IfcFlowSegment")

In [7]:
settings = ifcopenshell.geom.settings()
geom_library="hybrid-cgal-simple-opencascade"

In [8]:
# Getting the model representation context and sub context
model_context = ifcopenshell.util.representation.get_context(model, 'Model')
model_sub_context = ifcopenshell.util.representation.get_context(model, 'Model', 'Body', 'MODEL_VIEW')

In [9]:
# Creating the pipe representation
pipe_representation = ifcopenshell.api.geometry.add_mesh_representation(model, context=model_sub_context, vertices=[ v_matrix ], faces=[ f_matrix ])
pipe_representation

#5077591=IfcShapeRepresentation(#102,'Body','Brep',(#5077590))

OK. We have covered our bases from the previous implementation. Now, let's explore mapping and types. It seems like we can assign a representation to a ifc_class in the model. Here, instead of the `IfcFlowSegments`, we choose the `IfcFlowSegmentType` class.

In [10]:
element_type = ifcopenshell.api.root.create_entity(model, ifc_class="IfcFlowSegmentType")
element_type

#5077593=IfcFlowSegmentType('2X9BVqcKv0wBNmCRPmAkIB',#5077592,$,$,$,$,$,$,$)

To confirm this is different from `IfcFlowSegment`, here is what that data looks like.

In [11]:
element_type_test = ifcopenshell.api.root.create_entity(model, ifc_class="IfcFlowSegment")
element_type_test

#5077595=IfcFlowSegment('2oOylfBDH8QAUhGosoyYUh',#5077594,$,$,$,$,$,$)

Now, let's see what the existing representation looks like for this type. We use the same code from the previous implementation.

In [12]:
current_representation = ifcopenshell.util.representation.get_representation(element_type, context=model_sub_context)
print(current_representation)

None


Appears to not have a representation. Makes sense, since we would like to create the representation at this level. In my head, the workflow looks like this --> loop through all the geomtries in the model, remove the representation, and replace with a representation at the type level.

We start by assigning the pipe representation to the type level.

In [13]:
ifcopenshell.api.geometry.assign_representation(model, product=element_type, representation=pipe_representation)

Now, we should see this representation when we call the code from above.

In [14]:
current_representation = ifcopenshell.util.representation.get_representation(element_type, context=model_sub_context)
print(current_representation)

#5077591=IfcShapeRepresentation(#102,'Body','Brep',(#5077590))


Looks good. Next step is to remove the current representations that exist within the individual object level. We shall do so using the geom iterator from earlier. We could also apply the same transformation matrix from earlier. We start with the simple implementation, then if the scale issue persists, we shall work through another approach.

In [15]:
# settings.set(settings.USE_WORLD_COORDS, True)

# iterator = ifcopenshell.geom.iterator(
#     settings, model, include=pipes, geometry_library=geom_library)
# if iterator.initialize():
#     while True:
#         shape = iterator.get()
#         element = model.by_id(shape.id)

#         faces = ifcopenshell.util.shape.get_faces(shape.geometry)
#         verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

#         # Get the transformation matrix
#         T_matrix = get_transformation_matrix(verts)

#         # Replace Current Representation
#         current_representation = ifcopenshell.util.representation.get_representation(element, context=model_sub_context)
        
#         ifcopenshell.api.geometry.unassign_representation(model, product=element, representation=current_representation)
#         ifcopenshell.api.geometry.remove_representation(model, representation=current_representation, should_keep_named_profiles=False)

#         # Edit object placement
#         ifcopenshell.api.geometry.edit_object_placement(model, product=element, matrix=T_matrix)

#         ifcopenshell.api.type.assign_type(model, related_objects=[element], relating_type=element_type)

#         if not iterator.next():
#             break

In [16]:
# model.write('data/plumbing-test_3.ifc')

Scale issue persists. Let's work through a first principles approach.

## IFCMappedItem

Let's try to understand how the `IfcMappedItem` attribute works. We shall work through the basic example laid out in the docs and see if we can derive any insights from it. To start, let's create a blank project.

In [17]:
import ifcopenshell.api.root
import ifcopenshell.api.unit

In [18]:
ifc = ifcopenshell.file(schema='IFC4')

ifcopenshell.api.root.create_entity(ifc, ifc_class="IfcProject")

#1=IfcProject('2qzp3BZHHC8vt48AeAwNHU',$,$,$,$,$,$,$,$)

IfcProject is the root of all our data, and exists within every IFC file. Now, let's add units.

In [19]:
length = ifcopenshell.api.unit.add_si_unit(ifc, unit_type='LENGTHUNIT', prefix='MILLI')
ifcopenshell.api.unit.assign_unit(ifc, units=[length])

#3=IfcUnitAssignment((#2))

Now, we need to add our Representation context. We shall add one for the 'model' and one for the 'body'.

In [20]:
model3d = ifcopenshell.api.context.add_context(ifc, context_type='Model')

body = ifcopenshell.api.context.add_context(ifc, context_type='Model', context_identifier='Body', target_view='MODEL_VIEW', parent=model3d)

Let's sample some transformation matrices which we shall use for the `object_placement`. Let's get these using our premade function `get_transformation_matrix_2`, and our geometry iterator.

In [21]:
list_matrices = []
ctr = 0

iterator = ifcopenshell.geom.iterator(
    settings, model, include=pipes, geometry_library=geom_library)
if iterator.initialize():
    while True:
        shape = iterator.get()
        element = model.by_id(shape.id)

        faces = ifcopenshell.util.shape.get_faces(shape.geometry)
        verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

        # Get the transformation matrix
        T_matrix = get_transformation_matrix(verts)
        list_matrices.append(T_matrix)

        ctr += 1

        if not iterator.next() or ctr == 10:
            break

In [22]:
list_matrices

[array([[ 1.40000000e-02,  0.00000000e+00,  0.00000000e+00,
          2.20922357e+01],
        [ 0.00000000e+00,  1.95749545e+00, -1.21430643e-20,
          2.33414260e+01],
        [ 0.00000000e+00, -1.69785665e-18, -1.40000000e-02,
          3.82600000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]]),
 array([[ 7.10619232e-03,  1.73809812e-01, -4.30203582e-02,
          1.88070939e+01],
        [ 6.08399740e-02,  2.72897475e-17,  1.00496735e-02,
          4.12490571e+01],
        [-7.10619232e-03,  1.73809812e-01,  4.30203582e-02,
          4.39161654e-01],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]]),
 array([[ 0.        ,  0.        ,  0.0625    , 19.02684444],
        [ 0.0625    ,  0.        ,  0.        , 26.24188587],
        [ 0.        ,  0.18554361,  0.        ,  0.77945639],
        [ 0.        ,  0.        ,  0.        ,  1.        ]]),
 array([[-9.44895260e-03,  1.83834596e-01, -4.2492

We now have 10 matrices available to us. Let's now work on creating a mapped item. We have already created our pipe representation above, which used a mesh representation consisting of vertices and faces. Let's add this same representation to our new IfcProject.

First, we must create an entity.

In [23]:
element_type = ifcopenshell.api.root.create_entity(ifc, ifc_class='IfcFlowSegmentType')

element = ifcopenshell.api.root.create_entity(ifc, ifc_class="IfcFlowSegment")

In [24]:
test_vertices = [
    [(0.,0.,0.), (0.,2.,0.), (2.,2.,0.), (2.,0.,0.), (1.,1.,1.)] # A single mesh
]

test_faces = [
    [(0,1,2,3), (0,4,1), (1,4,2), (2,4,3), (3,4,0)] # A single mesh
]

In [25]:
pipe_representation_ifc = ifcopenshell.api.geometry.add_mesh_representation(ifc, context=body, vertices=test_vertices, faces=test_faces)
pipe_representation_ifc

#19=IfcShapeRepresentation(#9,'Body','Tessellation',(#18))

In [26]:
f_matrix

array([[ 0,  1,  3],
       [ 0,  3,  2],
       [ 2,  3,  5],
       [ 2,  5,  4],
       [ 4,  5,  7],
       [ 4,  7,  6],
       [ 6,  7,  9],
       [ 6,  9,  8],
       [ 8,  9, 11],
       [ 8, 11, 10],
       [10, 11, 13],
       [10, 13, 12],
       [12, 13, 15],
       [12, 15, 14],
       [14, 15, 17],
       [14, 17, 16],
       [16, 17, 19],
       [16, 19, 18],
       [18, 19, 21],
       [18, 21, 20],
       [20, 21, 23],
       [20, 23, 22],
       [22, 23, 25],
       [22, 25, 24],
       [24, 25, 27],
       [24, 27, 26],
       [26, 27, 29],
       [26, 29, 28],
       [28, 29, 31],
       [28, 31, 30],
       [30, 31, 33],
       [30, 33, 32],
       [32, 33, 35],
       [32, 35, 34],
       [34, 35, 37],
       [34, 37, 36],
       [36, 37, 39],
       [36, 39, 38],
       [38, 39, 41],
       [38, 41, 40],
       [40, 41, 43],
       [40, 43, 42],
       [42, 43, 45],
       [42, 45, 44],
       [44, 45, 47],
       [44, 47, 46],
       [46, 47, 49],
       [46, 4

In [27]:
f_matrix_tuple = [ tuple([ d.item() for d in row ]) for row in f_matrix ]
v_matrix_tuple = [ tuple([ d.item() for d in row ]) for row in v_matrix ]
f_matrix_tuple
v_matrix_tuple

[(0.0, 1.0, 1.0),
 (0.0, -1.0, 1.0),
 (0.19509, 1.0, 0.980785),
 (0.19509, -1.0, 0.980785),
 (0.382683, 1.0, 0.92388),
 (0.382683, -1.0, 0.923879),
 (0.55557, 1.0, 0.83147),
 (0.55557, -1.0, 0.83147),
 (0.707107, 1.0, 0.707107),
 (0.707107, -1.0, 0.707107),
 (0.83147, 1.0, 0.55557),
 (0.83147, -1.0, 0.55557),
 (0.92388, 1.0, 0.382683),
 (0.92388, -1.0, 0.382683),
 (0.980785, 1.0, 0.19509),
 (0.980785, -1.0, 0.19509),
 (1.0, 1.0, 0.0),
 (1.0, -1.0, -0.0),
 (0.980785, 1.0, -0.19509),
 (0.980785, -1.0, -0.19509),
 (0.92388, 1.0, -0.382683),
 (0.92388, -1.0, -0.382683),
 (0.83147, 1.0, -0.55557),
 (0.83147, -1.0, -0.55557),
 (0.707107, 1.0, -0.707107),
 (0.707107, -1.0, -0.707107),
 (0.55557, 1.0, -0.83147),
 (0.55557, -1.0, -0.83147),
 (0.382683, 1.0, -0.923879),
 (0.382683, -1.0, -0.92388),
 (0.19509, 1.0, -0.980785),
 (0.19509, -1.0, -0.980785),
 (0.0, 1.0, -1.0),
 (0.0, -1.0, -1.0),
 (-0.19509, 1.0, -0.980785),
 (-0.19509, -1.0, -0.980785),
 (-0.382683, 1.0, -0.923879),
 (-0.382683, -1

In [28]:
pipe_representation_ifc_2 = ifcopenshell.api.geometry.add_mesh_representation(ifc, context=body, vertices=[ v_matrix_tuple ], faces=[ f_matrix_tuple ])
pipe_representation_ifc_2

#146=IfcShapeRepresentation(#9,'Body','Tessellation',(#145))

As a note for future reference, we need to pass the vertex and face array as a list of tuples. There could be more meshes within the object, hence this argument also accepts a nested list.

Now, let's assign this representation to our pre-established `element_type`.

In [29]:
applied_rep = ifcopenshell.api.geometry.assign_representation(ifc, product=element_type, representation=pipe_representation_ifc_2)
applied_rep

And now, let's create our entities.

In [30]:
# list_elements = []

# for T_matrix in list_matrices:
#     element = ifcopenshell.api.root.create_entity(ifc, ifc_class="IfcFlowSegment")
#     list_elements.append(element)
#     ifcopenshell.api.geometry.edit_object_placement(ifc, product=element, matrix=T_matrix, is_si=True)

# ifcopenshell.api.type.assign_type(ifc, related_objects=list_elements, relating_type=element_type)

OK. Let's export this and see what we have. For reference, our scale factor is still missing here so we will need to add this later, but lets see if we at least have some instanced geometry.

In [31]:
# ifc.write('data/test_ifc.ifc')

Opening this in Blender, this is what we see.

![IfcMappedElement Test Case](../img/ifcmappedelement-test-case.png)

Looks good, this is what we expected. Our cylinders are in their correct locations. We also note that each one has the same mesh id, which will result in proper instancing when exporting to gltf. Now, we need to add the scale factor.

Naively, lets try adding our scale factor to the basic Transformation matrix we've created earlier. We can simply swap out our function `get_transformation_matrix_2` to `get_transformation_matrix`. 

AN ERROR. Seems like we had already used get_transformation_matrix from earlier, implying that our naive approach was very naive indeed. The Ifc transformation does not apply scaling, only rotation, and we need to apply scaling factors separately.

We do note in the documentation it mentions the following --> "# Objects are never scaled, so the scale factor of the matrix is always 1."

This could explain why our scale factor was essentially ignored. But now, we need to come up with a way to apply this factor to our unique instances.

We derive inspiration for this whole endeavour from the paper "The Unit Pipe: A Memory-Efficient Representation for Real-Time Visualization of Massive MEP Models", by Mikael Johansson and Mattias Roupé. In the paper, the authors explain that we must create a representation map, and call our instances as mapped items. This seems to be similar to what our Gemini suggested script was doing earlier, but in a much more convoluted and non ideal manner. Let's see if we can do better.

The paper references this graphic.

![Representation map graphic - credit Johansson et al](../img/ifcrepresentationmap-graphic.png)

If I understand this correctly, essentially we need to create a `IfcRepresentationMap`. This consists of a `IfcShapeRepresentation`, which in our case is the pipe representation from earlier.

Once we have this, we create an instance of this shaperepresentation by creating a new entity - `IfcMappedItem`, and together with a transformation operator, we can place our object in 3d space.

Let's create the representation map first. We derive this code directly from our earlier implementation.

In [32]:
# Create world origin placement for the map
origin = ifc.createIfcCartesianPoint((0.0, 0.0, 0.0))
axis_placement = ifc.createIfcAxis2Placement3D(origin)

# Create the map using the unit cylinder representation
rep_map = ifc.createIfcRepresentationMap(axis_placement, pipe_representation_ifc_2)
rep_map

#154=IfcRepresentationMap(#153,#146)

Now, lets derive our `IfcCartesianTransformationOperator3DnonUniform`. After quite some digging, it seems that this attribute takes direct values for the rotation vectors, scale vectors, and translation vectors. We can automatically derive these from our pre-established transformation matrix and scale vectors.

In [33]:
list_matrices = []
list_vectors = []
ctr = 0

iterator = ifcopenshell.geom.iterator(
    settings, model, include=pipes, geometry_library=geom_library)
if iterator.initialize():
    while True:
        shape = iterator.get()
        element = model.by_id(shape.id)

        faces = ifcopenshell.util.shape.get_faces(shape.geometry)
        verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

        # Get the transformation matrix
        T_matrix, scale_vec = get_transformation_matrix_2(verts)
        list_matrices.append(T_matrix)
        list_vectors.append(scale_vec)

        ctr += 1

        if not iterator.next() or ctr == 10:
            break

In [34]:
list_matrices

[array([[ 1.00000000e+00, -0.00000000e+00,  0.00000000e+00,
          2.20922357e+01],
        [ 0.00000000e+00,  1.00000000e+00, -8.67361738e-19,
          2.33414260e+01],
        [ 0.00000000e+00, -8.67361738e-19, -1.00000000e+00,
          3.82600000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]]),
 array([[ 1.15239785e-01,  7.07106781e-01, -6.97653060e-01,
          1.88070939e+01],
        [ 9.86630419e-01,  1.11022302e-16,  1.62973666e-01,
          4.12490571e+01],
        [-1.15239785e-01,  7.07106781e-01,  6.97653060e-01,
          4.39161654e-01],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]]),
 array([[-0.        ,  0.        ,  1.        , 19.02684444],
        [ 1.        ,  0.        ,  0.        , 26.24188587],
        [ 0.        ,  1.        ,  0.        ,  0.77945639],
        [ 0.        ,  0.        ,  0.        ,  1.        ]]),
 array([[-1.53487512e-01,  7.07106308e-01, -6.9024

In [35]:
list_vectors

[array([0.014     , 1.95749545, 0.014     ]),
 array([0.0616644 , 0.24580419, 0.0616644 ]),
 array([0.0625    , 0.18554361, 0.0625    ]),
 array([0.0615617 , 0.25998155, 0.0615617 ]),
 array([0.0625    , 0.29740275, 0.0625    ]),
 array([0.06175471, 0.28954241, 0.06175471]),
 array([0.06243067, 0.13499033, 0.06243067]),
 array([0.06242688, 0.22205151, 0.06242688]),
 array([0.06243875, 0.26018406, 0.06243875]),
 array([0.0625   , 0.3673269, 0.0625   ])]

In [36]:
pipe_rep = ifcopenshell.api.geometry.map_representation(ifc, representation=pipe_representation_ifc_2)
pipe_rep

#161=IfcShapeRepresentation(#9,'Body','MappedRepresentation',(#160))

In [37]:
list_elements = []

for i in range(len(list_matrices)):
    T_matrix = list_matrices[i]
    scale_vector = list_vectors[i]

    scale_x = scale_vector[0]
    scale_y = scale_vector[1]
    scale_z = scale_vector[2]

    translation_x = T_matrix[0,3]
    translation_y = T_matrix[1,3]
    translation_z = T_matrix[2,3]

    rotation_x = T_matrix[:-1, 0]
    rotation_y = T_matrix[:-1, 1]
    rotation_z = T_matrix[:-1, 2]

    # print([ d.item() for d in rotation_x ])

    translation_point = ifc.create_entity("IfcCartesianPoint", Coordinates=[translation_x.item(), translation_y.item(), translation_z.item()])

    x_rot = ifc.create_entity("IfcDirection", DirectionRatios=[ d.item() for d in rotation_x ])
    y_rot = ifc.create_entity("IfcDirection", DirectionRatios=[ d.item() for d in rotation_y ])
    z_rot = ifc.create_entity("IfcDirection", DirectionRatios=[ d.item() for d in rotation_z ])
    
    transformation_operator = ifc.create_entity(
        'IfcCartesianTransformationOperator3DnonUniform',
        Axis1 = x_rot,
        Axis2 = y_rot,
        Axis3 = z_rot,
        LocalOrigin = translation_point,
        Scale = scale_x,
        Scale2 = scale_y,
        Scale3 = scale_z
    )

    mapped_item = ifc.createIfcMappedItem(rep_map, transformation_operator)

In [38]:
ifc.write('data/test_ifc_2.ifc')

No luck. Unfortunately the IFC file format does not appear to be cooperating with us. Finding any kind of information online is also a hassle, the documentation is highly technical and in some cases, incomplete. We shall need to revisit this at a later time.

However, we're not done yet. While the IFC side of things appear to be stuck, lets see if we can export our file manually to gltf with these optimizations in place. Unlike IFC, the GLTF file format does allow for 4x4 transformation matrices on a isolated piece of geometry, so theoretically we could still achieve certain computational and memory efficiencies through this approach.

Once again, this notbook is getting quite large- so we shall pick this up in a new one.